# Live Demo — Real-Time Voice Cloning Detection (VoxDetect)

Two beats:
1. **REAL voice** — talk into the mic, live risk score via `score_audio`.
2. **CLONED voice** — play a pre-made team `clone_*.wav`, high risk alarm.

Ends with the numbers table (ACC / FPR / FNR / ROC-AUC) from our training runs. Run cells top-to-bottom.

In [ ]:
# @title 1. Mount Drive + clone repo + install deps + GPU check
from google.colab import drive
from pathlib import Path
import sys, os, subprocess

drive.mount("/content/drive")

REPO_URL = "https://github.com/io-PEAK/VoxDetect.git"
REPO_DIR = Path("/content/VoxDetect")
ML_BASE  = Path("/content/drive/MyDrive/VoxDetect/ml-core")
RESULTS_DIR = ML_BASE / "results"
CHECKPOINT_DIR = ML_BASE / "checkpoints"
for d in (RESULTS_DIR, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / "ml-core" / "src").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

SRC_PKG = REPO_DIR / "ml-core" / "src"
sys.path.insert(0, str(SRC_PKG))

# The LIVE demo scores in-memory audio, so we don't need the heavy training stack,
# just model inference + audio decoding + (optional) mic capture.
!pip install -q torch torchaudio transformers librosa soundfile resemblyzer numpy scipy sounddevice
%cd {REPO_DIR}

import torch
print("[gpu]", "CUDA" if torch.cuda.is_available() else "CPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
# GRANT MIC PERMISSION if the browser prompts: this demo can also run file-upload only.
print("\nIf the browser asked for microphone permission, ALLOW it. If not, use the upload path.")


In [ ]:
# @title 2. Load the detector (choice: pretrained base vs our fine-tuned checkpoint)
# Use the FINE-TUNED model if a sprint3+ checkpoint exists on Drive, else the Gustking base.
from detect import DetectionEngine

best = None
# Pick the newest fine-tuned checkpoint dir under Drive checkpoints/, if any.
cands = sorted([p for p in CHECKPOINT_DIR.iterdir() if (p/"config.json").exists()],
               key=lambda p: p.stat().st_mtime, reverse=True)
if cands:
    best = str(cands[0])
    print("using fine-tuned checkpoint:", best)
else:
    print("no fine-tuned checkpoint yet; using Gustking pretrained base")

eng = DetectionEngine(checkpoint=best)
# --- CALIBRATED THRESHOLD -------------------------------------------------
# Our fine-tuned model is GREAT at not flagging real voices (holdout fpr 0.0) but
# weak at catching clones (fnr 0.667 -> clone fused scores land under 70). Sprint0
# measured real=30 vs cloned=64 with the pretrained base. To give the CLONED beat a
# real chance, lower THRESH to split real vs clone. 50 sits between those. If the
# REAL beat ever mislabels, nudge up; if CLONED misses, nudge down.
THRESH = 50.0
print("\nloaded DetectionEngine. score_audio(wav, sr) ready.")


In [ ]:
# @title 3. LIVE beat — REAL voice (talk into the mic)
# Records ~3s from the mic, scores it in-memory via eng.score_audio, shows the verdict.
import numpy as np

def record_seconds(seconds=3.0, sr=16000):
    import sounddevice as sd
    print(f"Recording ~{seconds}s ... speak now")
    rec = sd.rec(int(seconds * sr), samplerate=sr, channels=1, dtype="float32")
    sd.wait()
    return rec[:, 0].astype("float32"), sr

try:
    wav, sr = record_seconds(3.0)
    r = eng.score_audio(wav, sr)
except Exception as e:
    print("mic unavailable ->", e)
    print("Falling back: attach a wav and score the file instead.")
    r = None

if r is not None:
    print("\n======================= REAL VOICE =======================")
    band = "REAL" if r["risk_score"] < THRESH else "CLONED"
    color = "\033[92m" if band == "REAL" else "\033[91m"
    print(color + f"\n   VERDICT: {band}   risk {r['risk_score']:.0f}%   (threshold {THRESH:.0f})" + "\033[0m")
    print("  signal model                   :", r["signals"]["model"])
    print("  signal prosody_anomaly         :", r["signals"]["prosody_anomaly"])
    print("  signal voiceprint_risk         :", r["signals"]["voiceprint_risk"])
    print("  signal context_risk            :", r["signals"]["context_risk"])
    print("===========================================================")


In [ ]:
# @title 4. CLONED beat — play a pre-made team clone clip
# Score one cloned clip from your team's dataset (or any fake wav). This is the
# 'CLONED' trigger for the demo. Drop a clone_*.wav into this session or reference
# one already on Drive / in /content/VoxDetect_data.
import glob

cands = (glob.glob("/content/VoxDetect_data/cloned/**/*.wav", recursive=True)
         + glob.glob("/content/*.wav") + glob.glob("/content/**/clone*.wav", recursive=True))
print("clone clips found:", len(cands))
if cands:
    path = cands[0]
    print("scoring:", path)
    import audio_utils
    wav, sr = audio_utils.load_audio(path)
    r = eng.score_audio(wav, sr)
    print("\n======================= CLONED VOICE =======================")
    band = "REAL" if r["risk_score"] < THRESH else "CLONED"
    color = "\033[92m" if band == "REAL" else "\033[91m"
    print(color + f"\n   VERDICT: {band}   risk {r['risk_score']:.0f}%   (threshold {THRESH:.0f})" + "\033[0m")
    print("  signal model                   :", r["signals"]["model"])
    print("  signal prosody_anomaly         :", r["signals"]["prosody_anomaly"])
    print("  signal voiceprint_risk         :", r["signals"]["voiceprint_risk"])
    print("  signal context_risk            :", r["signals"]["context_risk"])
    print("===========================================================")
else:
    print("No clone clip found. Add one, or reuse any clip below:")
    for p in glob.glob("/content/VoxDetect_data/**/*.wav", recursive=True)[:5]:
        print("   ", p)


In [ ]:
# @title 5. (Fallback) Score ANY uploaded wav, real or cloned
# If the mic failed or you don't have clips on disk, upload a file here and it's scored.
from google.colab import files
up = files.upload()
if up:
    fname = list(up.keys())[0]
    import audio_utils
    wav, sr = audio_utils.load_audio(fname)
    r = eng.score_audio(wav, sr)
    band = "REAL" if r["risk_score"] < THRESH else "CLONED"
    color = "\033[92m" if band == "REAL" else "\033[91m"
    print(color + f"\n   VERDICT: {band}   risk {r['risk_score']:.0f}%   (threshold {THRESH:.0f})" + "\033[0m")
    print("  signal model                   :", r["signals"]["model"])


In [ ]:
# @title 6. Numbers footer — back up the live demo (ACC / FPR / FNR / AUC)
# Read the folium-style results CSV (and any sprint JSONs) so the demo can end on the
# hard numbers that were logged when we trained on each dataset.
import json, glob

csv_path = RESULTS_DIR / "ablation_results.csv"
print("results CSV on Drive:", (RESULTS_DIR / "ablation_results.csv").exists())
if (RESULTS_DIR / "ablation_results.csv").exists():
    import pandas as pd
    df = pd.read_csv(csv_path)
    show = [c for c in ["variant","dataset","split","n_clips","accuracy","fpr","fnr","roc_auc","precision","recall","f1","threshold"] if c in df.columns]
    print(df[show].to_string(index=False))

print("\nsprint JSONs on Drive (results/*.json):")
for j in sorted(glob.glob(str(RESULTS_DIR / "*.json"))):
    try:
        d = json.load(open(j))
        print(f"  {Path(j).name}: mode={d.get('mode')} acc={d.get('accuracy')} fpr={d.get('fpr')} fnr={d.get('fnr')} auc={d.get('roc_auc')}")
    except Exception:
        pass
